# Re-scoring the embedding models on eval v3: semantic, then hybrid

One question in two steps, and neither costs an API call -- every corpus and query
vector is already cached from the earlier phases.

**Part 1** re-scores all seven models on the honestly-worded v3 questions.
**Part 2** checks the result survives production's hybrid RRF rather than the pure
semantic search of Part 1.

In [1]:
import gc
import hashlib
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from dotenv import load_dotenv

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
CHUNKS_PATH = PROJECT / "0. cleaned_data" / "chunks_articles.parquet"
CACHE_DIR = PROJECT / "eval" / "emb_cache"
load_dotenv(PROJECT / ".env")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
chunks = pd.read_parquet(CHUNKS_PATH)
chunks["article_number"] = chunks["article_number"].astype(str)

CORPUS_HASH = hashlib.md5(
    "".join(chunks["chunk_id"] + "\x00" + chunks["text"]).encode("utf-8")).hexdigest()[:12]

eval_v2 = pd.DataFrame([json.loads(l) for l in open(PROJECT / "eval" / "eval_v2.jsonl", encoding="utf-8")])
eval_v3 = pd.DataFrame([json.loads(l) for l in open(PROJECT / "eval" / "eval_v3.jsonl", encoding="utf-8")])

# the whole paired design rests on the ground truth being identical -- prove it, don't trust it
for f in ["question_id", "answer_law", "answer_article_numbers", "answer_language",
          "question_language", "difficulty", "question_category"]:
    assert eval_v2[f].tolist() == eval_v3[f].tolist(), f"v2/v3 differ on '{f}' -- pairing is invalid"

print(f"corpus hash {CORPUS_HASH} | {len(chunks)} articles")
print(f"v2 and v3 both {len(eval_v3)} questions, ground truth verified identical")
print("\ncached corpus vectors available for:")
for p in sorted(CACHE_DIR.glob(f"*__{CORPUS_HASH}__corpus.npy")):
    print("  ", p.name.split("__")[0])

corpus hash 88dbec8c163f | 531 articles
v2 and v3 both 60 questions, ground truth verified identical

cached corpus vectors available for:
   bge-m3
   cohere-embed-v4
   gemini-embedding-001
   multilingual-e5-large-instruct
   openai-text-embedding-3-large
   openai-text-embedding-3-small
   qwen3-embedding-0.6b


## Query encoders

Only queries need embedding. Formatting and pooling conventions are exactly those used in Phase 13 — a mismatch here would make the comparison meaningless rather than merely wrong.

In [2]:
QWEN_TASK = ("Given a question about Jordanian labor law or social security law, "
             "retrieve the law article that answers it")

MODELS = {
    "cohere-embed-v4":               dict(kind="cohere", model_id="embed-v4.0"),
    "openai-text-embedding-3-large": dict(kind="openai", model_id="text-embedding-3-large"),
    "openai-text-embedding-3-small": dict(kind="openai", model_id="text-embedding-3-small"),
    "gemini-embedding-001":          dict(kind="gemini", model_id="gemini-embedding-001"),
    "bge-m3":                        dict(kind="local", model_id="BAAI/bge-m3",
                                          fmt=lambda q: q, pool="cls", pad="right", maxlen=8192),
    "qwen3-embedding-0.6b":          dict(kind="local", model_id="Qwen/Qwen3-Embedding-0.6B",
                                          fmt=lambda q: f"Instruct: {QWEN_TASK}\nQuery: {q}",
                                          pool="last", pad="left", maxlen=8192),
    "multilingual-e5-large-instruct": dict(kind="local", model_id="intfloat/multilingual-e5-large-instruct",
                                           fmt=lambda q: f"query: {q}", pool="mean", pad="right", maxlen=512),
}


def _norm(x):
    return F.normalize(torch.tensor(x, dtype=torch.float32), p=2, dim=1)


def _retry(fn, label, retries=5):
    """The first attempt at this notebook died on a transient ConnectTimeout to Cohere,
    on the very first call, taking the whole run with it -- because this file was written
    without the retry and per-model guard that scripts/18. embeddings_evaluation_v2.ipynb had
    already been fixed to include. Same defect, repeated. Logs the message, not just the type."""
    last = None
    for attempt in range(retries):
        try:
            return fn()
        except Exception as e:
            last = e
            wait = min(30, 2 ** attempt)
            print(f"    {label} retry {attempt + 1}/{retries}: {type(e).__name__} -- "
                  f"{str(e)[:120]} -- waiting {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"{label} failed: {type(last).__name__}: {str(last)[:200]}")


def embed_queries(name, cfg, texts):
    kind = cfg["kind"]
    if kind == "cohere":
        import cohere
        c = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])
        r = _retry(lambda: c.embed(texts=texts, model=cfg["model_id"], input_type="search_query",
                                   embedding_types=["float"]), "cohere")
        return _norm(r.embeddings.float_)
    if kind == "openai":
        from openai import OpenAI
        c = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        r = _retry(lambda: c.embeddings.create(input=texts, model=cfg["model_id"]), "openai")
        return _norm([d.embedding for d in r.data])
    if kind == "gemini":
        from google import genai
        from google.genai import types
        c = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
        cf = types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
        out = []
        for i in range(0, len(texts), 16):
            r = _retry(lambda: c.models.embed_content(model=cfg["model_id"],
                                                      contents=texts[i:i + 16], config=cf), "gemini")
            out.extend(e.values for e in r.embeddings)
        return _norm(out)
    if kind == "local":
        from transformers import AutoTokenizer, AutoModel
        tok = AutoTokenizer.from_pretrained(cfg["model_id"])
        tok.padding_side = cfg["pad"]
        mdl = AutoModel.from_pretrained(
            cfg["model_id"], dtype=torch.float16 if DEVICE == "cuda" else torch.float32
        ).to(DEVICE).eval()
        outs = []
        with torch.no_grad():
            for i in range(0, len(texts), 16):
                batch = [cfg["fmt"](t) for t in texts[i:i + 16]]
                enc = tok(batch, padding=True, truncation=True,
                          max_length=cfg["maxlen"], return_tensors="pt").to(DEVICE)
                h = mdl(**enc).last_hidden_state
                if cfg["pool"] == "cls":
                    p = h[:, 0]
                elif cfg["pool"] == "last":
                    p = h[:, -1]
                else:
                    m = enc["attention_mask"].unsqueeze(-1).float()
                    p = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
                outs.append(F.normalize(p.float(), p=2, dim=1).cpu())
        del mdl, tok
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        return torch.cat(outs)
    raise ValueError(kind)

## Score

Same scorer as Phase 13: cosine top-10, rank of the first correct article, plus full coverage for the 8 multi-article questions.

In [3]:
TOP_K = 10


def score(query_emb, corpus_emb, ev):
    sims = query_emb @ corpus_emb.T
    topk = torch.topk(sims, k=TOP_K, dim=1)
    rows = []
    for qi, q in enumerate(ev.itertuples(index=False)):
        correct = {str(a) for a in q.answer_article_numbers}
        rank, found = None, set()
        for pos, idx in enumerate(topk.indices[qi].tolist(), start=1):
            c = chunks.iloc[idx]
            if (c["law"] == q.answer_law and c["language"] == q.answer_language
                    and c["article_number"] in correct):
                if rank is None:
                    rank = pos
                found.add(c["article_number"])
        rows.append({"question_id": q.question_id, "question_language": q.question_language,
                     "answer_law": q.answer_law, "difficulty": q.difficulty,
                     "rank": rank, "full_coverage": found == correct})
    return pd.DataFrame(rows)


import sys
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from evallib import metrics as _metrics

def metrics(df):
    return _metrics(df, include_n=False, full_coverage=True)


QUERY_HASH_V3 = hashlib.md5(
    "".join(eval_v3["question_id"] + "\x00" + eval_v3["question_text"]).encode("utf-8")).hexdigest()[:12]

FAILURES = {}
summaries, all_details = [], []
for name, cfg in MODELS.items():
    corpus_p = CACHE_DIR / f"{name}__{CORPUS_HASH}__corpus.npy"
    if not corpus_p.exists():
        print(f"skip {name} -- no cached corpus vectors")
        continue
    corpus_emb = torch.from_numpy(np.load(corpus_p))

    qp = CACHE_DIR / f"{name}__{QUERY_HASH_V3}__query.npy"
    if qp.exists():
        q_emb = torch.from_numpy(np.load(qp))
        note = "cached"
    else:
        t0 = time.time()
        try:
            q_emb = embed_queries(name, cfg, eval_v3["question_text"].tolist())
        except Exception as e:
            # one provider's outage must not cost the other six models their result
            FAILURES[name] = f"{type(e).__name__}: {str(e)[:200]}"
            print(f"{name:32s} FAILED -- {FAILURES[name]}")
            continue
        np.save(qp, q_emb.numpy())
        note = f"embedded in {time.time() - t0:.0f}s"

    det = score(q_emb, corpus_emb, eval_v3)
    det["model"] = name
    all_details.append(det)
    m = metrics(det)
    summaries.append({"model": name, **m})
    print(f"{name:32s} recall@1 {m['recall@1']:.3f}  recall@5 {m['recall@5']:.3f}  "
          f"MRR {m['mrr']:.3f}   ({note})")
    del corpus_emb, q_emb
    gc.collect()

if FAILURES:
    print("\nmodels with no v3 result:")
    for k, v in FAILURES.items():
        print(f"  {k}: {v}")

v3_summary = pd.DataFrame(summaries)
v3_details = pd.concat(all_details, ignore_index=True)
v3_summary.to_csv(PROJECT / "eval" / "embedding_eval_v3_summary.csv", index=False, encoding="utf-8-sig")
v3_details.to_csv(PROJECT / "eval" / "embedding_eval_v3_details.csv", index=False, encoding="utf-8-sig")

cohere-embed-v4                  recall@1 0.717  recall@5 1.000  MRR 0.828   (embedded in 2s)


openai-text-embedding-3-large    recall@1 0.733  recall@5 0.950  MRR 0.837   (embedded in 3s)


openai-text-embedding-3-small    recall@1 0.550  recall@5 0.883  MRR 0.694   (embedded in 2s)


gemini-embedding-001             recall@1 0.867  recall@5 1.000  MRR 0.922   (embedded in 4s)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

bge-m3                           recall@1 0.667  recall@5 0.967  MRR 0.803   (embedded in 12s)


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

qwen3-embedding-0.6b             recall@1 0.517  recall@5 0.833  MRR 0.657   (embedded in 4s)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

multilingual-e5-large-instruct   recall@1 0.633  recall@5 0.900  MRR 0.741   (embedded in 4s)


## Paired comparison: v2 vs v3

Same models, same corpus, same ground truth — only the question wording differs. The drop is therefore a measurement, not an inference.

In [4]:
v2_sum = pd.read_csv(PROJECT / "eval" / "embedding_eval_v2_summary.csv", encoding="utf-8-sig")
v2_sum = v2_sum[v2_sum.subset == "overall"].set_index("model")
v3_sum = v3_summary.set_index("model")

cmp = pd.DataFrame({
    "v2_recall@1": v2_sum["recall@1"], "v3_recall@1": v3_sum["recall@1"],
    "v2_recall@5": v2_sum["recall@5"], "v3_recall@5": v3_sum["recall@5"],
    "v2_mrr": v2_sum["mrr"], "v3_mrr": v3_sum["mrr"],
}).dropna()
cmp["d_recall@1"] = cmp["v3_recall@1"] - cmp["v2_recall@1"]
cmp["d_mrr"] = cmp["v3_mrr"] - cmp["v2_mrr"]
cmp = cmp.sort_values("v3_mrr", ascending=False)

pd.set_option("display.width", 170)
print("=" * 96)
print("PAIRED: identical ground truth, only the question wording changed")
print("=" * 96)
print(cmp.round(3).to_string())

print(f"\nmean recall@1 drop: {cmp['d_recall@1'].mean():+.3f}")
print(f"mean MRR drop:      {cmp['d_mrr'].mean():+.3f}")
print(f"\nmodels still at recall@5 = 1.000 -- v2: {(cmp['v2_recall@5'] == 1).sum()} of {len(cmp)}"
      f"  ->  v3: {(cmp['v3_recall@5'] == 1).sum()} of {len(cmp)}")

spread_v2 = cmp["v2_mrr"].max() - cmp["v2_mrr"].min()
spread_v3 = cmp["v3_mrr"].max() - cmp["v3_mrr"].min()
print(f"MRR spread across the field -- v2: {spread_v2:.3f}  ->  v3: {spread_v3:.3f} "
      f"({spread_v3 / spread_v2:.1f}x)")

PAIRED: identical ground truth, only the question wording changed


                                v2_recall@1  v3_recall@1  v2_recall@5  v3_recall@5  v2_mrr  v3_mrr  d_recall@1  d_mrr
model                                                                                                                
gemini-embedding-001                  0.967        0.867        1.000        1.000   0.981   0.922      -0.100 -0.058
openai-text-embedding-3-large         0.867        0.733        0.983        0.950   0.919   0.837      -0.133 -0.082
cohere-embed-v4                       0.950        0.717        1.000        1.000   0.975   0.828      -0.233 -0.147
bge-m3                                0.900        0.667        0.983        0.967   0.931   0.803      -0.233 -0.128
multilingual-e5-large-instruct        0.917        0.633        1.000        0.900   0.951   0.741      -0.283 -0.210
openai-text-embedding-3-small         0.817        0.550        0.950        0.883   0.876   0.694      -0.267 -0.183
qwen3-embedding-0.6b                  0.900        0.517

In [5]:
# How many questions does the field actually disagree on? That is the eval set's
# discriminating power -- in v2 it was 4 of 60, which is why the comparison was unusable.
piv = v3_details.pivot(index="question_id", columns="model", values="rank")
found = piv.notna().sum(axis=1)
contested = piv[(found > 0) & (found < piv.shape[1])]
never = piv[found == 0]

print(f"questions some models find and others miss entirely:  v3 = {len(contested)}  (v2 was 4)")
print(f"questions EVERY model misses:                         v3 = {len(never)}  (v2 was 0)")
if len(never):
    print("  ", never.index.tolist())

# not just "found vs missed" -- also where models rank the answer differently
disagree = piv.nunique(axis=1, dropna=False)
print(f"questions where models do NOT all agree on the rank:   v3 = {(disagree > 1).sum()} of {len(piv)}")

print("\nper-question ranks where the field disagrees (NaN = not in top 10):")
print(piv[disagree > 1].to_string())

questions some models find and others miss entirely:  v3 = 10  (v2 was 4)
questions EVERY model misses:                         v3 = 0  (v2 was 0)
questions where models do NOT all agree on the rank:   v3 = 48 of 60

per-question ranks where the field disagrees (NaN = not in top 10):
model        bge-m3  cohere-embed-v4  gemini-embedding-001  multilingual-e5-large-instruct  openai-text-embedding-3-large  openai-text-embedding-3-small  qwen3-embedding-0.6b
question_id                                                                                                                                                                   
q01             1.0              2.0                   4.0                             1.0                            2.0                            2.0                  10.0
q03             3.0              4.0                   1.0                             1.0                            4.0                            2.0                   5.0
q04            

## Verdict

### 1. About a quarter of the v2 scores were an artifact of phrasing

Every model fell, none by a little. Mean **recall@1 −0.233**, mean **MRR −0.155**, on identical ground truth with only the wording changed. That is a measurement, not an estimate: same 60 answers, same 531 articles, same cached corpus vectors.

`eval_v2` was measuring how closely a question copied its own answer at least as much as it was measuring retrieval.

### 2. The eval set works again

| | v2 | v3 |
|---|---|---|
| models at recall@5 = 1.000 | 4 of 7 | 2 of 7 |
| MRR spread across the field | 0.104 | **0.265** (2.5×) |
| questions some models find and others miss | 4 | **10** |
| questions where models disagree on rank | — | **48 of 60** |
| questions **every** model misses | 0 | **0** |

That last row matters most. The questions got harder without becoming unanswerable — every one is still findable by some model, so the difficulty is real rather than an artifact of a rewrite that broke the ground truth.

### 3. The ranking changed, and one change is significant

| model | v2 recall@1 | v3 recall@1 | v2 MRR | v3 MRR | Δ recall@1 |
|---|---|---|---|---|---|
| gemini-embedding-001 | 0.967 | **0.867** | 0.981 | **0.922** | −0.100 |
| openai-text-embedding-3-large | 0.867 | 0.733 | 0.919 | 0.837 | −0.133 |
| **cohere-embed-v4** *(incumbent)* | 0.950 | 0.717 | 0.975 | 0.828 | **−0.233** |
| bge-m3 | 0.900 | 0.667 | 0.931 | 0.803 | −0.233 |
| multilingual-e5-large-instruct | 0.917 | 0.633 | 0.951 | 0.741 | −0.283 |
| openai-text-embedding-3-small | 0.817 | 0.550 | 0.876 | 0.694 | −0.267 |
| qwen3-embedding-0.6b | 0.900 | 0.517 | 0.935 | 0.657 | **−0.383** |

Scores are deterministic and the questions are paired, so significance is testable directly with a sign test on which model ranks the answer first:

| comparison | A wins | B wins | p |
|---|---|---|---|
| **gemini vs cohere** | **12** | **3** | **0.035** |
| gemini vs openai-3-large | 12 | 4 | 0.077 |
| openai-3-large vs cohere | 9 | 8 | 1.000 |

**Gemini beats Cohere on evidence, not merely on table position.** OpenAI-3-large appears to overtake Cohere in the table, but the paired test says 9–8 — a coin flip. Ranking a leaderboard by a summary statistic invites exactly that misreading, which is why the pairwise test is here and not just the sorted table.

Qwen3-Embedding-0.6B is the cautionary result: 4th on v2, last on v3, the largest drop in the field. Its v2 placing was almost entirely phrasing.

### 4. The pre-committed decision rule does not fire — because the rule itself is flawed

Phase 13's rule: switch only if a challenger beats Cohere on **both recall@5 and MRR** by ≥ 0.05.

- MRR: Gemini **+0.094** — clears the bar nearly twice over.
- recall@5: **1.000 vs 1.000, tied.** Does not clear it.

Read literally, the rule says *do not switch*. But recall@5 is **saturated** — two models still sit at a perfect score, so it cannot express a difference between them however large that difference is. Choosing it as a primary metric was a mistake made in Phase 13, when it was *already* at ceiling for the top models.

Stating this plainly rather than quietly amending it: **a rule written to prevent post-hoc rationalisation must not itself be rewritten post-hoc.** The rule's purpose was to stop a one-question difference being sold as a discovery, and that purpose is not in tension with this result — the gap here is 12 questions to 3 at p = 0.035. But whether to override a pre-committed rule belongs to the project owner, not to the notebook that would benefit from the override.

**No change has been made. `src/retrieval.py` still uses Cohere `embed-v4.0`.**

### 5. The Gemini truncation caveat is now closed

Phase 13 flagged this as unverified. Now tested: `gemini-embedding-001` **silently truncates**. The longest article (3,279 Gemini tokens, over its 2,048 cap) is accepted with no error at all, as is an input six times longer.

Measured extent: **4 of 531 articles** exceed the cap — `ssl_ar_71`, `ssl_ar_64`, `ssl_en_71`, `ssl_en_64`. Only one of them, SSL Article 64, is ground truth for any question (q15), and **Gemini ranks it first regardless**. So the handicap is real but immaterial here — and Gemini won while carrying it.

It would matter more if the corpus later gained longer articles, because the failure is silent and would never surface as an error.

### Open items before any switch

1. **Hybrid, not semantic-only.** Production runs RRF(semantic, BM25); this is semantic alone. The cached vectors make that check free, and it should come first.
2. **Switching cost.** Gemini is 3072-dim against the current index's 1536 — a new Pinecone index, plus an embedder abstraction in `src/retrieval.py`.
3. **Deployment limits.** Cohere is already a funded production dependency. Gemini's free-tier request limits need checking against a live app before it goes in the request path — the same class of problem that removed Voyage from Phase 13.


---

## Part 2 -- does the result survive hybrid retrieval?

Part 1 ranked the models on pure semantic similarity. Production does not do that:
it fuses semantic ranks with BM25 keyword ranks via RRF. A semantic-only winner is
strong evidence, not proof.

There is also a confound to clear. The fusion weight `w` was fitted in Phase 6 on
clause chunks *with Cohere*, so it is not a neutral setting -- it was tuned in the
incumbent's favour. Part 2 therefore sweeps `w` across its whole range rather than
testing only the production value.

In [1]:
import sys
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from rank_bm25 import BM25Okapi

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
CACHE_DIR = PROJECT / "eval" / "emb_cache"
sys.path.insert(0, str(PROJECT))

# Reuse production's own Arabic normalisation and tokenizer rather than re-implementing
# them -- a subtly different tokenizer here would make this test measure the wrong thing.
from src.retrieval import (tokenize, RRF_K, RRF_W, BM25_K1, BM25_B, CANDIDATE_TOP_K)

chunks = pd.read_parquet(PROJECT / "0. cleaned_data" / "chunks_articles.parquet")
chunks["article_number"] = chunks["article_number"].astype(str)
eval_q = pd.DataFrame([json.loads(l) for l in open(PROJECT / "eval" / "eval_v3.jsonl", encoding="utf-8")])

CORPUS_HASH = hashlib.md5(
    "".join(chunks["chunk_id"] + "\x00" + chunks["text"]).encode("utf-8")).hexdigest()[:12]
QUERY_HASH = hashlib.md5(
    "".join(eval_q["question_id"] + "\x00" + eval_q["question_text"]).encode("utf-8")).hexdigest()[:12]

print(f"production constants -- RRF_K={RRF_K} RRF_W={RRF_W} "
      f"BM25_K1={BM25_K1} BM25_B={BM25_B} candidates={CANDIDATE_TOP_K}")
print(f"corpus {CORPUS_HASH} ({len(chunks)} articles) | queries {QUERY_HASH} ({len(eval_q)})")

bm25 = BM25Okapi([tokenize(t) for t in chunks["text"]], k1=BM25_K1, b=BM25_B)
print("BM25 index built from the same parquet production reads")

production constants -- RRF_K=5 RRF_W=0.85 BM25_K1=1.5 BM25_B=0.75 candidates=50
corpus 88dbec8c163f (531 articles) | queries 6e8e3718215d (60)
BM25 index built from the same parquet production reads


## Scoring

Identical fusion to `Retriever.retrieve()`: take the top `CANDIDATE_TOP_K` from each ranker, score each document `w/(k+rank_semantic) + (1-w)/(k+rank_keyword)`, sort, cut to 10.

In [2]:
TOP_K = 10
N = len(chunks)

# BM25 ranks depend only on the question, not on the embedding model -- compute once.
bm25_ranks = []
for q in eval_q["question_text"]:
    scores = bm25.get_scores(tokenize(q))
    bm25_ranks.append(np.argsort(-scores)[:CANDIDATE_TOP_K])


def rrf_ranked(sem_idx, kw_idx, w, k=RRF_K, top_k=TOP_K):
    s = {}
    for rank, i in enumerate(sem_idx, start=1):
        s[i] = s.get(i, 0.0) + w / (k + rank)
    for rank, i in enumerate(kw_idx, start=1):
        s[i] = s.get(i, 0.0) + (1 - w) / (k + rank)
    return [i for i, _ in sorted(s.items(), key=lambda kv: kv[1], reverse=True)[:top_k]]


def score_run(order_per_q):
    rows = []
    for qi, q in enumerate(eval_q.itertuples(index=False)):
        correct = {str(a) for a in q.answer_article_numbers}
        rank, found = None, set()
        for pos, idx in enumerate(order_per_q[qi], start=1):
            c = chunks.iloc[idx]
            if (c["law"] == q.answer_law and c["language"] == q.answer_language
                    and c["article_number"] in correct):
                if rank is None:
                    rank = pos
                found.add(c["article_number"])
        rows.append({"question_id": q.question_id, "rank": rank, "full_coverage": found == correct})
    return pd.DataFrame(rows)


import sys
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from evallib import metrics as _metrics

def metrics(df):
    return _metrics(df, include_n=False)


def semantic_ranks(model):
    corpus = torch.from_numpy(np.load(CACHE_DIR / f"{model}__{CORPUS_HASH}__corpus.npy"))
    queries = torch.from_numpy(np.load(CACHE_DIR / f"{model}__{QUERY_HASH}__query.npy"))
    sims = queries @ corpus.T
    return torch.topk(sims, k=CANDIDATE_TOP_K, dim=1).indices.numpy()


MODELS = ["gemini-embedding-001", "openai-text-embedding-3-large", "cohere-embed-v4",
          "bge-m3", "multilingual-e5-large-instruct", "openai-text-embedding-3-small",
          "qwen3-embedding-0.6b"]
MODELS = [m for m in MODELS if (CACHE_DIR / f"{m}__{QUERY_HASH}__query.npy").exists()]
SEM = {m: semantic_ranks(m) for m in MODELS}
print("loaded cached vectors for", len(MODELS), "models -- no API calls made")

loaded cached vectors for 7 models -- no API calls made


## Semantic-only vs hybrid, at production's `w = 0.85`

Plus a **BM25-only** row, which is the same for every model and shows what the keyword half contributes on its own.

In [3]:
bm25_only = score_run([list(r[:TOP_K]) for r in bm25_ranks])
print(f"BM25 alone (no embeddings):  recall@1 {metrics(bm25_only)['recall@1']:.3f}  "
      f"recall@5 {metrics(bm25_only)['recall@5']:.3f}  MRR {metrics(bm25_only)['mrr']:.3f}\n")

rows, hybrid_details = [], {}
for m in MODELS:
    sem_det = score_run([list(SEM[m][qi][:TOP_K]) for qi in range(len(eval_q))])
    hyb_order = [rrf_ranked(SEM[m][qi], bm25_ranks[qi], RRF_W) for qi in range(len(eval_q))]
    hyb_det = score_run(hyb_order)
    hybrid_details[m] = hyb_det
    s, h = metrics(sem_det), metrics(hyb_det)
    rows.append({"model": m,
                 "sem_recall@1": s["recall@1"], "hyb_recall@1": h["recall@1"],
                 "sem_recall@5": s["recall@5"], "hyb_recall@5": h["recall@5"],
                 "sem_mrr": s["mrr"], "hyb_mrr": h["mrr"],
                 "d_mrr": h["mrr"] - s["mrr"]})

cmp = pd.DataFrame(rows).sort_values("hyb_mrr", ascending=False)
pd.set_option("display.width", 170)
print("=" * 92)
print(f"SEMANTIC-ONLY vs HYBRID RRF (w={RRF_W}) on eval_v3")
print("=" * 92)
print(cmp.round(3).to_string(index=False))
cmp.to_csv(PROJECT / "eval" / "embedding_hybrid_v3_summary.csv", index=False, encoding="utf-8-sig")

BM25 alone (no embeddings):  recall@1 0.317  recall@5 0.600  MRR 0.437



SEMANTIC-ONLY vs HYBRID RRF (w=0.85) on eval_v3
                         model  sem_recall@1  hyb_recall@1  sem_recall@5  hyb_recall@5  sem_mrr  hyb_mrr  d_mrr
          gemini-embedding-001         0.867         0.883         1.000         1.000    0.922    0.931  0.008
 openai-text-embedding-3-large         0.733         0.733         0.950         0.950    0.837    0.838  0.001
               cohere-embed-v4         0.717         0.733         1.000         1.000    0.828    0.838  0.010
                        bge-m3         0.667         0.717         0.967         0.967    0.803    0.835  0.032
multilingual-e5-large-instruct         0.633         0.633         0.900         0.933    0.741    0.751  0.010
 openai-text-embedding-3-small         0.550         0.533         0.883         0.883    0.694    0.692 -0.002
          qwen3-embedding-0.6b         0.517         0.533         0.833         0.833    0.657    0.674  0.017


## Is the Gemini-over-Cohere result still there under hybrid?

Same paired sign test as Phase 15, now on hybrid ranks.

In [4]:
from math import comb


def sign_test(a, b, details):
    ha = (details[a]["rank"] == 1).values
    hb = (details[b]["rank"] == 1).values
    n10, n01 = int((ha & ~hb).sum()), int((~ha & hb).sum())
    n = n10 + n01
    p = 1.0 if n == 0 else min(1.0, 2 * sum(comb(n, k) for k in range(0, min(n10, n01) + 1)) / 2 ** n)
    return n10, n01, p


PAIRS = [("gemini-embedding-001", "cohere-embed-v4"),
         ("gemini-embedding-001", "openai-text-embedding-3-large"),
         ("openai-text-embedding-3-large", "cohere-embed-v4")]

print(f"{'comparison':62s} {'A':>4} {'B':>4} {'p':>8}")
for a, b in PAIRS:
    if a in hybrid_details and b in hybrid_details:
        w, l, p = sign_test(a, b, hybrid_details)
        print(f"{a + '  vs  ' + b:62s} {w:>4} {l:>4} {p:>8.4f}")

comparison                                                        A    B        p
gemini-embedding-001  vs  cohere-embed-v4                        11    2   0.0225
gemini-embedding-001  vs  openai-text-embedding-3-large          12    3   0.0352
openai-text-embedding-3-large  vs  cohere-embed-v4                8    8   1.0000


## Does the answer depend on `RRF_W`?

`w = 0.85` was fitted in Phase 6 on clause chunks with Cohere as the semantic ranker, so it is not a neutral setting for judging a different embedding model. Sweeping it from 0 (pure keyword) to 1 (pure semantic) shows whether the conclusion is an artifact of that one value.

In [5]:
WS = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0]
sweep = []
for m in MODELS:
    for w in WS:
        det = score_run([rrf_ranked(SEM[m][qi], bm25_ranks[qi], w) for qi in range(len(eval_q))])
        sweep.append({"model": m, "w": w, **metrics(det)})
sw = pd.DataFrame(sweep)
sw.to_csv(PROJECT / "eval" / "embedding_hybrid_v3_wsweep.csv", index=False, encoding="utf-8-sig")

print("MRR by RRF weight (w=0 pure BM25, w=1 pure semantic)")
print(sw.pivot(index="w", columns="model", values="mrr").round(3).to_string())

print("\nbest w per model, by MRR:")
best = sw.loc[sw.groupby("model")["mrr"].idxmax()][["model", "w", "recall@1", "recall@5", "mrr"]]
print(best.sort_values("mrr", ascending=False).round(3).to_string(index=False))

piv = sw.pivot(index="w", columns="model", values="mrr")
winner = piv.idxmax(axis=1)
print("\nwinner at each w:")
for w, mdl in winner.items():
    print(f"  w={w:<5} {mdl}")
print(f"\ndistinct winners across the sweep: {sorted(set(winner))}")

MRR by RRF weight (w=0 pure BM25, w=1 pure semantic)
model  bge-m3  cohere-embed-v4  gemini-embedding-001  multilingual-e5-large-instruct  openai-text-embedding-3-large  openai-text-embedding-3-small  qwen3-embedding-0.6b
w                                                                                                                                                                       
0.00    0.437            0.437                 0.437                           0.437                          0.437                          0.437                 0.437
0.20    0.502            0.506                 0.507                           0.503                          0.488                          0.490                 0.497
0.40    0.638            0.642                 0.687                           0.637                          0.661                          0.591                 0.627
0.50    0.744            0.736                 0.797                           0.725                  

## Verdict: the Phase 15 result survives hybrid, and survives the weight sweep

### The ranking does not change when BM25 joins

| model | semantic MRR | hybrid MRR (w=0.85) | Δ |
|---|---|---|---|
| **gemini-embedding-001** | 0.922 | **0.931** | +0.008 |
| openai-text-embedding-3-large | 0.837 | 0.838 | +0.001 |
| **cohere-embed-v4** *(incumbent)* | 0.828 | 0.838 | +0.010 |
| bge-m3 | 0.803 | 0.835 | +0.032 |
| multilingual-e5-large-instruct | 0.741 | 0.751 | +0.010 |
| openai-text-embedding-3-small | 0.694 | 0.692 | −0.002 |
| qwen3-embedding-0.6b | 0.657 | 0.674 | +0.017 |

Paired sign test on hybrid ranks — **gemini beats cohere 11–2, p = 0.0225**, slightly *stronger* than the semantic-only 12–3 / p = 0.035. Gemini also now separates from openai-3-large (12–3, p = 0.035). And openai-3-large vs cohere is **8–8, p = 1.000** — still a dead tie, confirming the semantic-only finding that the table ordering between those two means nothing.

### Gemini wins at every weight that uses the embeddings at all

| w | 0.0 | 0.2 | 0.4 | 0.6 | 0.8 | 0.85 | 1.0 |
|---|---|---|---|---|---|---|---|
| gemini | 0.437 | **0.507** | **0.687** | **0.864** | **0.931** | **0.931** | **0.922** |
| cohere | 0.437 | 0.506 | 0.642 | 0.761 | 0.844 | 0.838 | 0.828 |

This was the confound worth testing: `w = 0.85` was fitted in Phase 6 on clause chunks *with Cohere as the semantic ranker*, so using it to judge Gemini was not neutral. It turns out not to matter — **Gemini leads at every w from 0.2 to 1.0**, and the margin *widens* as the embedding gets more weight.

The decisive comparison is Gemini at Cohere's *own* best setting: at w = 0.80, where Cohere peaks at 0.844, Gemini scores **0.931**. Cohere's best configuration anywhere in the sweep (0.844) is still below Gemini's worst from w = 0.6 upward (0.864).

*(Reading note: at w = 0.0 all seven models are identical at 0.437 — that arm is pure BM25 and ignores embeddings entirely. The "winner at w=0.0" line reports bge-m3 only because `idxmax` breaks a seven-way tie alphabetically. It is a tie, not a win.)*

### BM25 is contributing very little here

**BM25 alone: recall@1 0.317, recall@5 0.600, MRR 0.437** — far below every embedding model. And fusing it in moves MRR by between −0.002 and +0.032, with Gemini gaining just +0.008.

Two consequences worth carrying forward:

1. **Hybrid is nearly free upside but not a rescue.** It never hurt the strong models meaningfully and helped the weak ones most (bge-m3 +0.032). The retrieval quality of this system is essentially the embedding model's quality.
2. **This weakens the earlier concern that natural phrasing would favour BM25.** It did not — v3's questions avoid statute vocabulary, which is exactly what BM25 needs, so the keyword half got *less* useful, not more.

### `w = 0.85` is no longer the best setting

Every model peaks at **w = 0.80 or 0.70**, none at 0.85. That confirms the RRF re-tune pending since Phase 12a is genuinely warranted — but the prize is small (Cohere 0.838 → 0.844; Gemini unchanged at 0.931), and it should be done *after* the embedding model is settled, not before, since the optimum moves with the model.

### Bottom line

Every check that could have overturned the Phase 15 result has now been run and none did:

- ✅ holds under production's exact hybrid configuration
- ✅ holds across the entire RRF weight range, including Cohere's own optimum
- ✅ statistically significant on paired tests in both semantic-only (p = 0.035) and hybrid (p = 0.0225)
- ✅ Gemini achieves it while *silently truncating* 4 articles that Cohere reads in full

The one metric that says otherwise — recall@5 tied at 1.000 — is saturated and cannot express a difference. **The evidence supports switching to `gemini-embedding-001`.**

Still not done unilaterally, and still not done in this notebook: switching means a new 3072-dim Pinecone index, an embedder abstraction in `src/retrieval.py`, and checking Gemini's production rate limits — the failure mode that removed Voyage in Phase 13. `src/retrieval.py` remains on Cohere `embed-v4.0`.
